In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report
import joblib


In [3]:
df = pd.read_csv("hand_data.csv")

print("Dataset shape:", df.shape)

Dataset shape: (4912, 64)


In [5]:
X = df.drop("label", axis=1).values
y = df["label"].values

# Encode labels (A,B,C → 0,1,2...)
encoder = LabelEncoder()
y = encoder.fit_transform(y)

num_classes = len(np.unique(y))
y = to_categorical(y, num_classes)

print("Number of classes:", num_classes)

Number of classes: 28


In [7]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Train: (3929, 63)
Validation: (491, 63)
Test: (492, 63)


In [11]:

model = Sequential([
    Dense(256, activation='relu', input_shape=(63,)),
    Dropout(0.3),

    Dense(256, activation='relu'),
    Dropout(0.3),

    Dense(128, activation='relu'),
    Dropout(0.2),

    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Layer (type)             ┃ Output Shape      ┃   Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ dense (Dense)            │ (None, 256)       │    16,384 │
├──────────────────────────┼───────────────────┼───────────┤
│ dropout (Dropout)        │ (None, 256)       │         0 │
├──────────────────────────┼───────────────────┼───────────┤
│ dense_1 (Dense)          │ (None, 256)       │    65,792 │
├──────────────────────────┼───────────────────┼───────────┤
│ dropout_1 (Dropout)      │ (None, 256)       │         0 │
├──────────────────────────┼───────────────────┼───────────┤
│ dense_2 (Dense)          │ (None, 128)       │    32,896 │
├──────────────────────────┼───────────────────┼───────────┤
│ dropout_2 (Dropout)      │ (None, 128)       │         0 │
├──────────────────────────┼───────────────────┼───────────┤
│ dense_3 (Dense)          │ (None, 28)        │     3,612 │
└──────────────────────────┴───────────────────┴───────────┘

 Total params: 118,684 (463.61 KB)

 Trainable params: 118,684 (463.61 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5
)


In [15]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4703 - loss: 1.8978 - val_accuracy: 0.8961 - val_loss: 0.5490 - learning_rate: 0.0010
Epoch 2/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8521 - loss: 0.5055 - val_accuracy: 0.9593 - val_loss: 0.1859 - learning_rate: 0.0010
Epoch 3/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9323 - loss: 0.2490 - val_accuracy: 0.9756 - val_loss: 0.1061 - learning_rate: 0.0010
Epoch 4/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9534 - loss: 0.1729 - val_accuracy: 0.9796 - val_loss: 0.0950 - learning_rate: 0.0010
Epoch 5/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9598 - loss: 0.1456 - val_accuracy: 0.9898 - val_loss: 0.0960 - learning_rate: 0.0010
Epoch 6/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9715 - loss: 0.1039 - val_accuracy: 0.9756 - val_loss: 0.1435 - learning_rate: 0.0010
Epoch 7/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9717 - loss: 0.

In [16]:
loss, acc = model.evaluate(X_test, y_test)
print(f"\nFinal Test Accuracy: {acc*100:.2f}%")

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9919 - loss: 0.0588 

Final Test Accuracy: 99.19%


In [19]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes))


16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      1.00      1.00        20
           2       1.00      1.00      1.00        24
           3       1.00      1.00      1.00        24
           4       1.00      1.00      1.00        19
           5       1.00      0.89      0.94        18
           6       1.00      1.00      1.00        22
           7       1.00      1.00      1.00        18
           8       1.00      1.00      1.00        16
           9       1.00      1.00      1.00        13
          10       1.00      1.00      1.00        18
          11       1.00      1.00      1.00        16
          12       1.00      1.00      1.00         9
          13       1.00      1.00      1.00         1
          14       1.00      1.00      1.00        19
          15       0.94      1.00      0.97        16
          16      

In [24]:
model.save("hand_landmark_model.h5")
joblib.dump(scaler, "scaler.pkl")
np.save("label_classes.npy", encoder.classes_)

print("\nModel, scaler, and label classes saved successfully!")


Model, scaler, and label classes saved successfully!
